# Stage A — Deterministic Rule-Based Classification

This notebook implements the first stage of the hybrid GMRID risk-classification pipeline.

Stage A uses deterministic rules, keywords, phrases, and selected metadata to classify clear cases without embeddings or an LLM.

Records that cannot be classified confidently will later be routed to Stage B.

In [2]:
import pandas as pd
import numpy as np
import ast
import re
from pathlib import Path

In [3]:
DATA_DIR = Path("../data")

train_poc = pd.read_csv(DATA_DIR / "train_poc.csv")
test_poc = pd.read_csv(DATA_DIR / "test_poc.csv")

print("Train shape:", train_poc.shape)
print("Test shape:", test_poc.shape)

Train shape: (3322, 16)
Test shape: (819, 16)


In [4]:
# CSV does not preserve Python lists or booleans reliably
def restore_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    return ast.literal_eval(value)

train_poc["true_risks"] = train_poc["true_risks"].apply(restore_list)
test_poc["true_risks"] = test_poc["true_risks"].apply(restore_list)

In [5]:
print(type(train_poc.loc[0, "true_risks"]))
print(train_poc.loc[0, "true_risks"])

<class 'list'>
['weather_disruption', 'port_operational_disruption']


In [6]:
# Normalize metadata booleans
def to_bool(value):
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true", "1", "yes"
    }


for col in ["maritime_label", "contains_port_info"]:
    if col in train_poc.columns:
        train_poc[col] = train_poc[col].apply(to_bool)

    if col in test_poc.columns:
        test_poc[col] = test_poc[col].apply(to_bool)

In [7]:
print(train_poc.columns.tolist())

['id', 'Headline', 'Details', 'Severity', 'Region', 'Datetime', 'lat', 'lon', 'maritime_label', 'found_ports', 'contains_port_info', 'Category', 'Summarized_label', 'true_risks', 'input_text', 'normalized_text']


In [8]:
train_poc[
    [
        "Headline",
        "Category",
        "true_risks",
        "normalized_text"
    ]
].head()

,Headline,Category,true_risks,normalized_text
0,Severe winds caused brief suspension at Port o...,Port Disruption,"[weather_disruption, port_operational_disruption]",severe winds caused brief suspension at port o...
1,USA: 'Fridays for Future' climate change prote...,Protest / Riot,[labor_strike_disruption],usa: 'fridays for future' climate change prote...
2,UPDATE - Indonesia: Severe winds damage infras...,"Roadway Closure / Disruption, Flooding, Severe...",[weather_disruption],update - indonesia: severe winds damage infras...
3,UPDATE 1 - Refrigerated container import capac...,"Port Disruption, Cargo Disruption, Port Conges...",[port_operational_disruption],update 1 - refrigerated container import capac...
4,Marine wind warning issued for Port of Sydney ...,Weather Advisory,[weather_disruption],marine wind warning issued for port of sydney ...


In [9]:
RISK_TAXONOMY = {
    "weather_disruption": {
        "name": "Weather Disruption",
        "description": (
            "Disruption to trade, transport, ports, logistics, or infrastructure "
            "caused by severe weather such as storms, flooding, high winds, "
            "cyclones, or similar weather-related hazards."
        )
    },

    "natural_disaster": {
        "name": "Natural Disaster",
        "description": (
            "Disruption caused by natural disasters such as earthquakes, "
            "tsunamis, volcanic activity, landslides, or other geological hazards."
        )
    },

    "port_operational_disruption": {
        "name": "Port Operational Disruption",
        "description": (
            "Disruption to normal port or cargo operations caused by congestion, "
            "capacity limitations, operational delays, cargo disruption, "
            "or reduced terminal efficiency."
        )
    },

    "port_closure": {
        "name": "Port Closure",
        "description": (
            "Full or partial closure, suspension, or shutdown of a port, terminal, "
            "pier, berth, or related maritime facility."
        )
    },

    "labor_strike_disruption": {
        "name": "Labor / Strike Disruption",
        "description": (
            "Disruption caused by worker strikes, industrial action, labor disputes, "
            "walkouts, or related workforce actions affecting transport or logistics."
        )
    },

    "maritime_security_navigation_disruption": {
        "name": "Maritime Security / Navigation Disruption",
        "description": (
            "Disruption or increased operational risk to maritime transport caused "
            "by maritime advisories, piracy, security threats, navigation restrictions, "
            "or waterway closures and disruptions."
        )
    }
}

In [10]:
for risk_id, info in RISK_TAXONOMY.items():
    print(risk_id, "->", info["name"])

weather_disruption -> Weather Disruption
natural_disaster -> Natural Disaster
port_operational_disruption -> Port Operational Disruption
port_closure -> Port Closure
labor_strike_disruption -> Labor / Strike Disruption
maritime_security_navigation_disruption -> Maritime Security / Navigation Disruption


In [11]:
# checking how many records are multi-label

train_poc["num_true_risks"] = train_poc["true_risks"].apply(len)

train_poc["num_true_risks"].value_counts().sort_index()

num_true_risks
1    2371
2     871
3      77
4       3
Name: count, dtype: int64

## Stage A Design

Stage A uses deterministic evidence such as:

- strong phrases
- individual keywords
- port-related metadata
- maritime metadata
- weighted rule scores

Each risk receives a score.

A risk is predicted only when its score crosses a confidence threshold.

Because GMRID events may represent more than one risk, Stage A supports multi-label predictions.

train_poc.head()

In [12]:
train_poc.head()

,id,Headline,Details,Severity,Region,Datetime,lat,lon,maritime_label,found_ports,contains_port_info,Category,Summarized_label,true_risks,input_text,normalized_text,num_true_risks
0,552,Severe winds caused brief suspension at Port o...,Local sources reported that operations at Pier...,Minor,South Africa,2018-12-28 05:15:00,-29.88386,31.01766,True,['durban'],True,Port Disruption,Weather,"[weather_disruption, port_operational_disruption]",Severe winds caused brief suspension at Port o...,severe winds caused brief suspension at port o...,2
1,2973,USA: 'Fridays for Future' climate change prote...,Protests against climate change are anticipate...,Minor,United States,2019-11-27 07:00:00,NaN,NaN,False,['new york'],True,Protest / Riot,Worker Strike,[labor_strike_disruption],USA: 'Fridays for Future' climate change prote...,usa: 'fridays for future' climate change prote...,1
2,6,UPDATE - Indonesia: Severe winds damage infras...,Severe winds have downed billboards and trees ...,Moderate,Indonesia,2017-04-19 09:10:00,-6.91264,107.65700,False,['jakarta'],True,"Roadway Closure / Disruption, Flooding, Severe...",Weather,[weather_disruption],UPDATE - Indonesia: Severe winds damage infras...,update - indonesia: severe winds damage infras...,1
3,4537,UPDATE 1 - Refrigerated container import capac...,Updated sources indicate that terminals at the...,Moderate,China,2020-02-03 21:57:00,38.98074,117.74600,True,['tianjin'],True,"Port Disruption, Cargo Disruption, Port Conges...",Administrative Issue,[port_operational_disruption],UPDATE 1 - Refrigerated container import capac...,update 1 - refrigerated container import capac...,1
4,3771,Marine wind warning issued for Port of Sydney ...,Industry sources indicate that the Bureau of M...,Minor,Australia,2020-05-21 09:25:00,-33.97373,151.21680,False,['sydney'],True,Weather Advisory,Weather,[weather_disruption],Marine wind warning issued for Port of Sydney ...,marine wind warning issued for port of sydney ...,1


In [13]:
print(train_poc["maritime_label"].value_counts(dropna=False))
print()
print(train_poc["contains_port_info"].value_counts(dropna=False))

maritime_label
False    2385
True      937
Name: count, dtype: int64

contains_port_info
True    3322
Name: count, dtype: int64


In [14]:
# dropping contains_port_info 
train_poc = train_poc.drop(columns=["contains_port_info"], errors="ignore")
test_poc = test_poc.drop(columns=["contains_port_info"], errors="ignore")

In [15]:
STAGE_A_RULES = {

    "weather_disruption": {
        "strong_phrases": [
            "severe weather",
            "adverse weather",
            "inclement weather",
            "weather advisory",
            "weather advisories",
            "weather warning",
            "weather warnings",
            "tropical cyclone",
            "tropical storm",
            "storm surge",
            "thunderstorm warning",
            "severe thunderstorm",
            "high winds",
            "strong wind",
            "strong winds",
            "severe wind",
            "severe winds",
            "heavy rainfall",
            "heavy rain",
            "torrential rain",
            "flash flooding",
            "flooding",
            "typhoon",
            "hurricane",
            "cyclone",
            "tornado"
        ],

        "keywords": [
            "storm",
            "flood",
            "rainfall",
            "weather",
            "winds",
            "hail",
            "blizzard"
        ],

        "context": None,
        "threshold": 3
    },

    "natural_disaster": {
        "strong_phrases": [
            # These single terms are sufficiently specific to count
            # as strong evidence in this maritime/logistics dataset.
            "earthquake",
            "tsunami",
            "volcanic eruption",
            "volcano",
            "landslide"
        ],

        "keywords": [
            "magnitude",
            "seismic",
            "volcanic"
        ],

        "context": None,
        "threshold": 3
    },

    "port_operational_disruption": {
        "strong_phrases": [
            # Direct congestion and disruption descriptions
            "port congestion",
            "terminal congestion",
            "berth congestion",
            "berthing congestion",
            "cargo congestion",
            "port disruption",
            "cargo disruption",
            "operational disruption",
            "operations disrupted",
            "congestion",
            "backlog",

            # Backlogs, delays, and waiting-time descriptions
            "container backlog",
            "cargo backlog",
            "vessel backlog",
            "berthing delay",
            "berthing delays",
            "berth delay",
            "berth delays",
            "port delay",
            "port delays",
            "terminal delay",
            "terminal delays",
            "vessel delay",
            "vessel delays",
            "shipping delay",
            "shipping delays",
            "cargo delay",
            "cargo delays",
            "operational delays",
            "waiting time",
            "waiting times",
            "vessels waiting",
            "ships waiting",
            "waiting for a berth",
            "waiting for berth",
            "berthing time",
            "dwelling time",

            # Capacity and terminal-efficiency descriptions
            "capacity shortage",
            "capacity shortages",
            "capacity constraint",
            "capacity constraints",
            "yard remains full",
            "yard is full",
            "high yard density",
            "reduced capacity",
            "low productivity",
            "port operations normal",
            "waterside operations",
            "gate closures",

            # Common wording found in the GMRID records
            "impact port operations",
            "impacts port operations",
            "affect port operations",
            "affects port operations",
            "disrupt port operations",
            "disrupts port operations",
            "disrupt operations at",
            "disrupts operations at",
            "impact operations at",
            "impacts operations at",
            "affect operations at",
            "affects operations at",
            "shipping cancellations",
            "omit port",
            "skips port"
        ],

        "keywords": [
            "delay",
            "delays",
            "delayed",
            "waiting",
            "capacity",
            "shortage",
            "disruption",
            "disrupted"
        ],

        "context": "port",
        "threshold": 3
    },

    "port_closure": {
        "strong_phrases": [
            "port closure",
            "closure of the port",
            "closure of port",
            "port closed",
            "port closes",
            "port to close",
            "port will close",
            "close the port",
            "close port",
            "terminal closure",
            "terminal closed",
            "terminal closes",
            "terminal to close",
            "port operations suspended",
            "port suspends operations",
            "port suspends all operations",
            "suspend operations at the port",
            "operations suspended at port",
            "operations suspended at the port",
            "terminal operations suspended",
            "terminals suspend operations",
            "port shutdown",
            "port shut down",
            "shut down port",
            # "port reopens",
            # "port reopened",
            # "port has reopened",
            "port bans"
        ],

        "keywords": [
            "closure",
            "closed",
            "closes",
            "suspended",
            "suspend",
            "shutdown",
            "standstill",
            "reopens",
            "reopened"
        ],

        "context": "port",
        "threshold": 4
    },

    "labor_strike_disruption": {
        "strong_phrases": [
            # "strike" is direct enough to be strong evidence.
            "strike",
            "strikes",
            "industrial action",
            "industrial actions",
            "labor dispute",
            "labour dispute",
            "work stoppage",
            "work stoppages",
            "walkout",
            "walkouts",
            "workers protest",
            "worker protest",
            "dockworkers protest",
            "longshore workers"
        ],

        "keywords": [
            "striking",
            "dockworker",
            "dockworkers",
            "longshoremen",
            "union",
            "unions",
            "labor",
            "labour",
            "workers",
            "truckers",
            "transporters"
        ],

        "context": None,
        "threshold": 3
    },

    "maritime_security_navigation_disruption": {
        "strong_phrases": [
            # Direct maritime security or navigation evidence
            "maritime advisory",
            "maritime security",
            "navigation warning",
            "navigation warnings",
            "navigation restriction",
            "navigation restrictions",
            "waterway closure",
            "waterway disruption",

            # Waterway and vessel-traffic restrictions
            "ship channel closes",
            "ship channel closed",
            "ship channel reopens",
            "ship channel reopened",
            "shipping channel closes",
            "shipping channel closed",
            "shipping channel reopens",
            "vessel traffic suspended",
            "vessel traffic halted",
            "vessel traffic reopened",
            "vessel traffic resumes",
            "river vessel traffic",
            "restricted berthing",

            # Piracy and vessel-security incidents
            "piracy",
            "pirates",
            "pirate attack",
            "armed robbers",
            "robbers board",
            "robbers boarded",
            "robbery aboard",
            "robbery reported aboard",
            "ship attacked",
            "vessel attacked",
            "tanker attacked",
            "carrier attacked",

            # Common maritime schedule-advisory wording
            "blank sailing",
            "blanks week"
        ],

        # Generic words such as ship, vessel, traffic, or channel occur in
        # many non-security records. Keep this Stage A class conservative
        # and let Stage B handle cases without a direct phrase.
        "keywords": [],

        "context": "maritime",
        "threshold": 3
    }
}

In [ ]:


for risk_id, rules in STAGE_A_RULES.items():

    print(risk_id)
    print("Strong phrases:", len(rules["strong_phrases"]))
    print("Keywords:", len(rules["keywords"]))
    print("Context:", rules["context"])
    print("Threshold:", rules["threshold"])
    print("-" * 50)

weather_disruption
Strong phrases: 26
Keywords: 7
Context: None
Threshold: 3
--------------------------------------------------
natural_disaster
Strong phrases: 5
Keywords: 3
Context: None
Threshold: 3
--------------------------------------------------
port_operational_disruption
Strong phrases: 64
Keywords: 8
Context: port
Threshold: 3
--------------------------------------------------
port_closure
Strong phrases: 25
Keywords: 9
Context: port
Threshold: 4
--------------------------------------------------
labor_strike_disruption
Strong phrases: 14
Keywords: 11
Context: None
Threshold: 3
--------------------------------------------------
maritime_security_navigation_disruption
Strong phrases: 35
Keywords: 0
Context: maritime
Threshold: 3
--------------------------------------------------


### Rule-Based Scoring

Each of the six risks is scored independently.

For a given risk:

- Strong phrase match = **+3 points**
- Keyword match = **+1 point**
- Relevant port or maritime context = **+1 point**

Context can support existing textual evidence, but context alone cannot
create a Stage A prediction.

Each risk may define its own confidence threshold.

In [ ]:

# Cache compiled patterns because the same terms are applied to every row.
TERM_PATTERN_CACHE = {}


def get_term_pattern(term):

    if term not in TERM_PATTERN_CACHE:

        # Lookarounds prevent a term such as "union" from matching inside
        # a larger unrelated word. Spaces are allowed to match any
        # normalized whitespace between phrase words.
        escaped_term = re.escape(
            term.lower()
        ).replace(
            r"\ ",
            r"\s+"
        )

        TERM_PATTERN_CACHE[term] = re.compile(
            r"(?<![a-z0-9])"
            + escaped_term
            + r"(?![a-z0-9])"
        )

    return TERM_PATTERN_CACHE[term]


def term_is_present(text, term):

    pattern = get_term_pattern(
        term
    )

    return pattern.search(text) is not None


def find_distinct_matches(text, terms):

    matched_terms = []

    # Check longer phrases first. This prevents "cyclone" and
    # "tropical cyclone" from being counted as two separate pieces
    # of evidence when they refer to the same text span.
    ordered_terms = sorted(
        dict.fromkeys(terms),
        key=lambda term: (
            -len(term),
            term
        )
    )

    for term in ordered_terms:

        if not term_is_present(text, term):
            continue

        contained_in_existing_match = any(
            term_is_present(existing_match, term)
            for existing_match in matched_terms
        )

        if not contained_in_existing_match:
            matched_terms.append(
                term
            )

    return matched_terms


PORT_CONTEXT_TERMS = [
    "port",
    "terminal",
    "pier",
    "berth",
    "harbor",
    "harbour",
    "dock"
]


MARITIME_CONTEXT_TERMS = [
    "ship",
    "shipping",
    "vessel",
    "maritime",
    "marine",
    "waterway",
    "channel",
    "strait",
    "anchorage",
    "tanker",
    "carrier",
    "barge",
    "port",
    "harbor",
    "harbour"
]


def has_any_term(text, terms):

    return any(
        term_is_present(text, term)
        for term in terms
    )


def get_context_bonus(text, context, row):

    if context == "port":

        return int(
            has_any_term(
                text,
                PORT_CONTEXT_TERMS
            )
        )

    if context == "maritime":

        has_maritime_metadata = to_bool(
            row.get(
                "maritime_label",
                False
            )
        )

        has_maritime_text = has_any_term(
            text,
            MARITIME_CONTEXT_TERMS
        )

        return int(
            has_maritime_metadata
            or has_maritime_text
        )

    return 0


def calculate_rule_score(text, rule_config, row=None):

    if pd.isna(text):
        text = ""
    else:
        text = str(text).lower()

    matched_phrases = find_distinct_matches(
        text,
        rule_config["strong_phrases"]
    )

    possible_keywords = find_distinct_matches(
        text,
        rule_config["keywords"]
    )

    matched_keywords = []

    for keyword in possible_keywords:

        # Do not count a keyword again when it is already part of a
        # matched strong phrase. For example, "weather" should not add
        # another point after "weather warning" has matched.
        already_in_phrase = any(
            term_is_present(phrase, keyword)
            for phrase in matched_phrases
        )

        if not already_in_phrase:
            matched_keywords.append(
                keyword
            )

    score = (
        3 * len(matched_phrases)
        + len(matched_keywords)
    )

    context_bonus = 0

    # Metadata or contextual wording can support textual evidence,
    # but it cannot create a prediction by itself.
    if score > 0 and row is not None:

        context_bonus = get_context_bonus(
            text=text,
            context=rule_config.get("context"),
            row=row
        )

        score += context_bonus

    return {
        "score": score,
        "matched_phrases": matched_phrases,
        "matched_keywords": matched_keywords,
        "context_bonus": context_bonus
    }


In [ ]:


def get_stage_a_scores(row):

    text = row.get(
        "normalized_text",
        ""
    )

    results = {}

    for risk_id, rule_config in STAGE_A_RULES.items():

        results[risk_id] = calculate_rule_score(
            text=text,
            rule_config=rule_config,
            row=row
        )

    return results

In [19]:
example_row = train_poc.iloc[0]

print("Headline:")
print(example_row["Headline"])

print("\nTrue risks:")
print(example_row["true_risks"])

print("\nStage A scores:")
get_stage_a_scores(example_row)


Headline:
Severe winds caused brief suspension at Port of Durban terminals

True risks:
['weather_disruption', 'port_operational_disruption']

Stage A scores:


{'weather_disruption': {'score': 6,
  'matched_phrases': ['severe winds', 'strong winds'],
  'matched_keywords': [],
  'context_bonus': 0},
 'natural_disaster': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0},
 'port_operational_disruption': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0},
 'port_closure': {'score': 2,
  'matched_phrases': [],
  'matched_keywords': ['suspended'],
  'context_bonus': 1},
 'labor_strike_disruption': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0},
 'maritime_security_navigation_disruption': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0}}

In [20]:
def classify_stage_a(row):

    risk_scores = get_stage_a_scores(
        row
    )

    predictions = []

    for risk_id, evidence in risk_scores.items():

        threshold = STAGE_A_RULES[
            risk_id
        ].get(
            "threshold",
            3
        )

        if evidence["score"] >= threshold:
            predictions.append(
                risk_id
            )

    prediction_evidence = {
        risk_id: risk_scores[risk_id]
        for risk_id in predictions
    }

    return {
        "predictions": predictions,
        "evidence": prediction_evidence,
        "route_to_stage_b": len(predictions) == 0
    }


train_stage_a_results = train_poc.apply(
    classify_stage_a,
    axis=1
)

test_stage_a_results = test_poc.apply(
    classify_stage_a,
    axis=1
)


train_poc["stage_a_predictions"] = train_stage_a_results.apply(
    lambda result: result["predictions"]
)

train_poc["stage_a_evidence"] = train_stage_a_results.apply(
    lambda result: result["evidence"]
)

train_poc["route_to_stage_b"] = train_stage_a_results.apply(
    lambda result: result["route_to_stage_b"]
)


test_poc["stage_a_predictions"] = test_stage_a_results.apply(
    lambda result: result["predictions"]
)

test_poc["stage_a_evidence"] = test_stage_a_results.apply(
    lambda result: result["evidence"]
)

test_poc["route_to_stage_b"] = test_stage_a_results.apply(
    lambda result: result["route_to_stage_b"]
)

In [21]:
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer


RISK_IDS = list(
    RISK_TAXONOMY.keys()
)


def evaluate_stage_a(dataframe, split_name):

    label_binarizer = MultiLabelBinarizer(
        classes=RISK_IDS
    )

    true_matrix = label_binarizer.fit_transform(
        dataframe["true_risks"]
    )

    predicted_matrix = label_binarizer.transform(
        dataframe["stage_a_predictions"]
    )

    print(
        f"{split_name} classification report"
    )

    print(
        classification_report(
            true_matrix,
            predicted_matrix,
            target_names=[
                RISK_TAXONOMY[risk_id]["name"]
                for risk_id in RISK_IDS
            ],
            zero_division=0
        )
    )

    micro_precision, micro_recall, micro_f1, _ = (
        precision_recall_fscore_support(
            true_matrix,
            predicted_matrix,
            average="micro",
            zero_division=0
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            true_matrix,
            predicted_matrix,
            average="macro",
            zero_division=0
        )
    )

    covered_mask = dataframe[
        "stage_a_predictions"
    ].apply(
        lambda predictions: len(predictions) > 0
    )

    exact_match = np.mean(
        [
            set(true_risks) == set(predicted_risks)
            for true_risks, predicted_risks in zip(
                dataframe["true_risks"],
                dataframe["stage_a_predictions"]
            )
        ]
    )

    coverage = covered_mask.mean()

    print(
        "Micro precision:",
        round(micro_precision, 4)
    )

    print(
        "Micro recall:",
        round(micro_recall, 4)
    )

    print(
        "Micro F1:",
        round(micro_f1, 4)
    )

    print(
        "Macro precision:",
        round(macro_precision, 4)
    )

    print(
        "Macro recall:",
        round(macro_recall, 4)
    )

    print(
        "Macro F1:",
        round(macro_f1, 4)
    )

    print(
        "Exact multi-label match:",
        round(exact_match, 4)
    )

    print(
        "Stage A coverage:",
        round(coverage, 4)
    )

    print(
        "Rows routed to Stage B:",
        int((~covered_mask).sum())
    )

    return {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "exact_match": exact_match,
        "coverage": coverage
    }

In [22]:
# Use training results while changing the rule lists.
train_stage_a_metrics = evaluate_stage_a(
    train_poc,
    "Training"
)

Training classification report
                                           precision    recall  f1-score   support

                       Weather Disruption       0.98      0.75      0.85      1300
                         Natural Disaster       0.85      0.93      0.89       103
              Port Operational Disruption       0.88      0.64      0.74      1394
                             Port Closure       0.54      0.27      0.36       355
                Labor / Strike Disruption       0.96      0.76      0.85       740
Maritime Security / Navigation Disruption       0.97      0.21      0.34       464

                                micro avg       0.91      0.62      0.74      4356
                                macro avg       0.86      0.59      0.67      4356
                             weighted avg       0.90      0.62      0.72      4356
                              samples avg       0.71      0.65      0.67      4356

Micro precision: 0.9102
Micro recall: 0.6214
Micro F1

In [23]:
def get_label_error_rows(
    dataframe,
    risk_id,
    error_type
):

    true_mask = dataframe[
        "true_risks"
    ].apply(
        lambda risks: risk_id in risks
    )

    predicted_mask = dataframe[
        "stage_a_predictions"
    ].apply(
        lambda risks: risk_id in risks
    )

    if error_type == "false_positive":

        error_mask = (
            predicted_mask
            & ~true_mask
        )

    elif error_type == "false_negative":

        error_mask = (
            true_mask
            & ~predicted_mask
        )

    else:
        raise ValueError(
            "error_type must be "
            "'false_positive' or 'false_negative'"
        )

    columns_to_show = [
        "id",
        "Headline",
        "Category",
        "Summarized_label",
        "true_risks",
        "stage_a_predictions",
        "stage_a_evidence"
    ]

    return dataframe.loc[
        error_mask,
        columns_to_show
    ].copy()

In [24]:
port_closure_false_positives = get_label_error_rows(
    train_poc,
    risk_id="port_closure",
    error_type="false_positive"
)

print(
    "Port closure false positives:",
    len(port_closure_false_positives)
)

port_closure_false_positives.head(20)

Port closure false positives: 82


,id,Headline,Category,Summarized_label,true_risks,stage_a_predictions,stage_a_evidence
94,1098,Alexandria port reopens to vessels traffic fol...,Port Disruption,Weather,"[weather_disruption, port_operational_disruption]",[port_closure],"{'port_closure': {'score': 4, 'matched_phrases..."
139,1496,Egypt: Alexandria Port and El Dekheila Port cl...,"Port Disruption, Weather Advisory",Weather,"[weather_disruption, port_operational_disruption]","[weather_disruption, port_closure]","{'weather_disruption': {'score': 6, 'matched_p..."
140,1608,Heavy wind likely to disrupt Port of Nagoya,Port Disruption,Weather,"[weather_disruption, port_operational_disruption]",[port_closure],"{'port_closure': {'score': 4, 'matched_phrases..."
214,5645,UPDATE 1 - Strong winds likely to impact port ...,Port Disruption,Weather,"[weather_disruption, port_operational_disruption]","[weather_disruption, port_operational_disrupti...","{'weather_disruption': {'score': 9, 'matched_p..."
234,1418,Container ships delayed at Port of Charleston ...,Port Disruption,Weather,"[weather_disruption, port_operational_disruption]","[weather_disruption, port_operational_disrupti...","{'weather_disruption': {'score': 3, 'matched_p..."
245,3792,Minor congestion reported at Guangzhou Oceanga...,Port Congestion,Weather,"[weather_disruption, port_operational_disruption]","[weather_disruption, port_operational_disrupti...","{'weather_disruption': {'score': 3, 'matched_p..."
262,1970,Peru Updated reports indicate the strike has...,Cargo Transportation Strike,Worker Strike,[labor_strike_disruption],"[port_operational_disruption, port_closure, la...","{'port_operational_disruption': {'score': 7, '..."
268,4694,UPDATE 2 - Strong wind likely to impact operat...,Port Disruption,Weather,"[weather_disruption, port_operational_disruption]","[weather_disruption, port_operational_disrupti...","{'weather_disruption': {'score': 3, 'matched_p..."
313,4184,Strong winds expected to disrupt Port of Yokoh...,Port Disruption,Weather,"[weather_disruption, port_operational_disruption]","[weather_disruption, port_closure]","{'weather_disruption': {'score': 3, 'matched_p..."
362,2941,UPDATE: Waiting times remain at 0.5 day at Por...,Port Congestion,Administrative Issue,[port_operational_disruption],"[port_operational_disruption, port_closure]","{'port_operational_disruption': {'score': 4, '..."


In [25]:
port_closure_false_negatives = get_label_error_rows(
    train_poc,
    risk_id="port_closure",
    error_type="false_negative"
)

print(
    "Port closure false negatives:",
    len(port_closure_false_negatives)
)

port_closure_false_negatives.head(20)

Port closure false negatives: 260


,id,Headline,Category,Summarized_label,true_risks,stage_a_predictions,stage_a_evidence
10,627,Strong winds to disrupt port operations across...,"Port Disruption,Port Closure",Weather,"[weather_disruption, port_operational_disrupti...","[weather_disruption, port_operational_disruption]","{'weather_disruption': {'score': 6, 'matched_p..."
41,432,Operations suspended at Dammam Port,Port Closure,Weather,"[weather_disruption, port_closure]",[weather_disruption],"{'weather_disruption': {'score': 3, 'matched_p..."
45,4636,UPDATE 2 - Alexandria Port Authority announces...,Port Closure,Weather,"[weather_disruption, port_closure]",[weather_disruption],"{'weather_disruption': {'score': 3, 'matched_p..."
52,2888,UPDATE: Port of Tokyo suspends all operations ...,Port Closure,Weather,"[weather_disruption, port_closure]",[],{}
62,4003,Port of Taichung to close between Jan 24-26 fo...,Port Closure,Administrative Issue,[port_closure],[],{}
63,3665,Inbound channels closed at Port of Ningbo due ...,Port Closure,Weather,"[weather_disruption, port_closure]",[],{}
70,5358,Port of San Antonio closes on October 12 due t...,Port Closure,Weather,"[weather_disruption, port_closure]",[],{}
75,875,UPDATE: Operations resume at Port of Southampt...,Port Closure,Weather,"[weather_disruption, port_closure]","[weather_disruption, port_operational_disruption]","{'weather_disruption': {'score': 3, 'matched_p..."
111,4006,Port of Tianjin closes due to dense fog on Jan...,Port Closure,Weather,"[weather_disruption, port_closure]",[],{}
120,2854,UPDATE: Operations resume at Port of Southampt...,Port Closure,Weather,"[weather_disruption, port_closure]",[],{}


In [26]:
# Run the test evaluation only after the Stage A rules are finalized.
# Do not repeatedly tune the rules using test-set errors.
test_stage_a_metrics = evaluate_stage_a(
    test_poc,
    "Test"
)

Test classification report
                                           precision    recall  f1-score   support

                       Weather Disruption       0.98      0.76      0.85       341
                         Natural Disaster       0.94      0.89      0.91        35
              Port Operational Disruption       0.87      0.67      0.76       351
                             Port Closure       0.57      0.34      0.43        85
                Labor / Strike Disruption       0.98      0.76      0.86       179
Maritime Security / Navigation Disruption       1.00      0.24      0.38        97

                                micro avg       0.91      0.66      0.76      1088
                                macro avg       0.89      0.61      0.70      1088
                             weighted avg       0.91      0.66      0.75      1088
                              samples avg       0.75      0.69      0.70      1088

Micro precision: 0.9141
Micro recall: 0.6553
Micro F1: 0.

In [27]:
stage_a_metric_comparison = pd.DataFrame(
    [
        train_stage_a_metrics,
        test_stage_a_metrics
    ],
    index=[
        "Training",
        "Test"
    ]
)

stage_a_metric_comparison.round(4)

,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,exact_match,coverage
Training,0.9102,0.6214,0.7386,0.8629,0.5917,0.6707,0.5379,0.7529
Test,0.9141,0.6553,0.7634,0.8896,0.6088,0.6983,0.5665,0.7900


In [28]:
import json
from pathlib import Path


OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


stage_a_train_output = pd.DataFrame({
    "id": train_poc["id"],
    "stage_a_predictions": train_poc[
        "stage_a_predictions"
    ].apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False
        )
    )
})


stage_a_test_output = pd.DataFrame({
    "id": test_poc["id"],
    "stage_a_predictions": test_poc[
        "stage_a_predictions"
    ].apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False
        )
    )
})


stage_a_train_output.to_csv(
    OUTPUT_DIR / "stage_a_train_predictions.csv",
    index=False
)

stage_a_test_output.to_csv(
    OUTPUT_DIR / "stage_a_test_predictions.csv",
    index=False
)


print(
    "Stage A prediction files exported successfully."
)

Stage A prediction files exported successfully.
